# Fast Configuration Testing for Demographic Simulation

This notebook uses **10% stratified sampling** of GSS respondents to quickly test different:
- Demographic persona configurations (which fields to include)
- Prompt formats
- Model attention implementations

Only uses variables present in **all 3 GSS years** (2021, 2022, 2024).

In [ ]:
import sys; sys.path.insert(0, "..")
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
import numpy as np
import torch
import gc
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

from model_utils import load_model, extract_heads_batched, get_model_info
from run_gss_pca import compute_all_head_metrics_pca

## 1. Configuration — Edit this cell to test different setups

In [ ]:
# =====================================================================
# MODEL
# =====================================================================
MODEL_PATH = '/project/jevans/maxzhuyt/models/gemma-2-9b-it'
MODEL_NAME = MODEL_PATH.split('/')[-1]
ATTN_IMPL = 'sdpa'

# =====================================================================
# SAMPLING
# =====================================================================
SAMPLE_FRAC = 0.005         # fraction of respondents per topic
SAMPLE_SEED = 42

# Stratification columns (respondents are binned on these before sampling)
STRATIFY_COLS = ['polviews', 'age_bin', 'degree', 'race', 'born', 'sex', 'sexornt', 'rincome', 'region']

# =====================================================================
# DEMOGRAPHIC PERSONA FIELDS
# Choose which fields to include in the persona prompt.
# Set to None to use ALL fields present in all 3 years.
# =====================================================================
# Option A: Core 11 fields (original config)
DEMO_FIELDS_CORE = [
    'age', 'sex', 'race', 'degree', 'marital', 'relig', 'region',
    'rincome', 'wrkstat', 'childs', 'polviews'
]

# Option B: Expanded set with more context
DEMO_FIELDS_EXPANDED = [
    'age', 'sex', 'race', 'degree', 'marital', 'relig', 'region',
    'rincome', 'wrkstat', 'childs', 'polviews', 'born', 'educ',
    'health', 'sibs', 'sexornt'
]

# Option C: All 83 demographic fields present in all 3 years
DEMO_FIELDS_ALL = None  # will be loaded from CSV

# >>> SELECT WHICH SET TO USE <<<
ACTIVE_DEMO_FIELDS = DEMO_FIELDS_ALL

# =====================================================================
# LIFESTYLE FIELDS (optional, set to True to include)
# =====================================================================
INCLUDE_LIFESTYLE = False

# =====================================================================
# PROMPT FORMAT
# =====================================================================
SYSTEM_MSG = 'You are simulating the views of an American.'

# Prompt template: {profile} and {question} are filled in per respondent
# Format A: Original
PROMPT_FMT_A = '{profile}\n\nSurvey question: {question}'
# Format B: More conversational
PROMPT_FMT_B = 'Given the following background about a person:\n{profile}\n\nHow would they answer: {question}'
# Format C: Minimal
PROMPT_FMT_C = '{profile}\n\n{question}'

# >>> SELECT WHICH FORMAT TO USE <<<
ACTIVE_PROMPT_FMT = PROMPT_FMT_B

BATCH_SIZE = 90
MAX_LENGTH = 512
PCA_DIMS = [15]
# DATA PATHS
GSS_DATA = '../../data/gss/gss_2021_2024.csv'
STATA_2024 = '../../data/gss/GSS2024.dta'
STATA_2022 = '../../data/gss/GSS2022.dta'
DEMO_CSV = '../../question_lists/gss_demographic_variables.csv'
LIFESTYLE_CSV = '../../question_lists/gss_politicized_lifestyle_variables.csv'
PUBLIC_TOPICS = '../../question_lists/public_issues.csv'
PUBLIC_POL = '../../data/polarization/public_issues_polarization.csv'
PRIVATE_TOPICS = '../../question_lists/private_life.csv'
PRIVATE_POL = '../../data/polarization/private_life_polarization.csv'

print('Configuration ready.')
print(f'  Demo fields: {ACTIVE_DEMO_FIELDS if ACTIVE_DEMO_FIELDS else "ALL (83)"}')
print(f'  Include lifestyle: {INCLUDE_LIFESTYLE}')
print(f'  Sample fraction: {SAMPLE_FRAC}')
print(f'  Prompt format: {ACTIVE_PROMPT_FMT[:80]}...')

Configuration ready.
  Demo fields: ALL (83)
  Include lifestyle: False
  Sample fraction: 0.005
  Prompt format: Given the following background about a person:
{profile}

How would they answer:...


## 2. Load field definitions & filter to all-3-year variables

In [ ]:
# Variables present in all 3 GSS years (2021, 2022, 2024)
DEMO_VARS_ALL_YEARS = {
    'age', 'agekdbrn', 'babies', 'born', 'childs', 'degree', 'denom', 'denom16',
    'dipged', 'divorce', 'earnrs', 'educ', 'evwork', 'famdif16', 'family16',
    'granborn', 'health', 'hompop', 'hrs1', 'hrs2', 'incom16', 'income',
    'indus10', 'madeg', 'maeduc', 'maind10', 'major1', 'maocc10', 'marital',
    'mawrkgrw', 'mawrkslf', 'mobile16', 'numemps', 'occ10', 'othlang',
    'othlang1', 'othlang2', 'padeg', 'paeduc', 'paind10', 'paocc10', 'parborn',
    'partfull', 'pawrkslf', 'polviews', 'posslq', 'posslqy', 'preteen', 'race',
    'reg16', 'region', 'relig', 'relig16', 'res16', 'rincome', 'sex', 'sexornt',
    'sibs', 'spdeg', 'spden', 'speduc', 'spevwork', 'sphrs1', 'sphrs2',
    'spind10', 'spklang', 'spocc10', 'sprel', 'spwrkslf', 'spwrksta', 'teens',
    'unemp', 'unrelat', 'weekswrk', 'widowed', 'wksub', 'wksubs', 'wksup',
    'wksups', 'wrkslf', 'wrkstat', 'xnorcsiz', 'yousup'
}

LIFESTYLE_VARS_ALL_YEARS = {
    'attend', 'compuse', 'fear', 'helpoth', 'hivtest', 'hunt', 'news', 'obey',
    'owngun', 'partners', 'pistol', 'popular', 'pray', 'reborn', 'relactiv',
    'rifle', 'rowngun', 'sexfreq', 'shotgun', 'socbar', 'socfrend', 'socommun',
    'socrel', 'spanking', 'thnkself', 'tvhours', 'union', 'vetyears', 'vote16',
    'webmob', 'workhard', 'wrkgovt1', 'wrkslf', 'xmovie'
}

# Load field metadata (ConciseDescription, CodeMapping)
demo_df = pd.read_csv(DEMO_CSV)
demo_labels = dict(zip(demo_df['VariableName'], demo_df['ConciseDescription']))

lifestyle_df = pd.read_csv(LIFESTYLE_CSV)
lifestyle_labels = dict(zip(lifestyle_df['VariableName'], lifestyle_df['ConciseDescription']))

# Build active field list
if ACTIVE_DEMO_FIELDS is None:
    active_fields = sorted(DEMO_VARS_ALL_YEARS)
else:
    # Keep only fields that are in all 3 years
    active_fields = [f for f in ACTIVE_DEMO_FIELDS if f in DEMO_VARS_ALL_YEARS]
    missing = [f for f in ACTIVE_DEMO_FIELDS if f not in DEMO_VARS_ALL_YEARS]
    if missing:
        print(f'WARNING: Dropped fields not in all 3 years: {missing}')

# Build label lookup
all_labels = {}
all_labels.update(demo_labels)

# Add lifestyle fields if requested
lifestyle_fields = set()
if INCLUDE_LIFESTYLE:
    lifestyle_fields = LIFESTYLE_VARS_ALL_YEARS.copy()
    for var in sorted(lifestyle_fields):
        if var not in [f for f in active_fields]:
            active_fields.append(var)
    all_labels.update(lifestyle_labels)

# Ensure all active fields have labels
for f in active_fields:
    if f not in all_labels:
        all_labels[f] = f  # fallback to variable name

print(f'Active demographic fields: {len([f for f in active_fields if f not in lifestyle_fields])}')
print(f'Active lifestyle fields: {len([f for f in active_fields if f in lifestyle_fields])}')
print(f'Total active fields: {len(active_fields)}')
print(f'Fields: {active_fields}')

Active demographic fields: 83
Active lifestyle fields: 0
Total active fields: 83
Fields: ['age', 'agekdbrn', 'babies', 'born', 'childs', 'degree', 'denom', 'denom16', 'dipged', 'divorce', 'earnrs', 'educ', 'evwork', 'famdif16', 'family16', 'granborn', 'health', 'hompop', 'hrs1', 'hrs2', 'incom16', 'income', 'indus10', 'madeg', 'maeduc', 'maind10', 'major1', 'maocc10', 'marital', 'mawrkgrw', 'mawrkslf', 'mobile16', 'numemps', 'occ10', 'othlang', 'othlang1', 'othlang2', 'padeg', 'paeduc', 'paind10', 'paocc10', 'parborn', 'partfull', 'pawrkslf', 'polviews', 'posslq', 'posslqy', 'preteen', 'race', 'reg16', 'region', 'relig', 'relig16', 'res16', 'rincome', 'sex', 'sexornt', 'sibs', 'spdeg', 'spden', 'speduc', 'spevwork', 'sphrs1', 'sphrs2', 'spind10', 'spklang', 'spocc10', 'sprel', 'spwrkslf', 'spwrksta', 'teens', 'unemp', 'unrelat', 'weekswrk', 'widowed', 'wksub', 'wksubs', 'wksup', 'wksups', 'wrkslf', 'wrkstat', 'xnorcsiz', 'yousup']


## 3. Build code maps from Stata files

In [ ]:
print('Loading value labels from Stata files...')

df_num_24 = pd.read_stata(STATA_2024, convert_categoricals=False)
df_cat_24 = pd.read_stata(STATA_2024, convert_categoricals=True)

df_num_22 = pd.read_stata(STATA_2022, convert_categoricals=False)
reader_22 = pd.io.stata.StataReader(STATA_2022)
vl_22 = reader_22.value_labels()
var_to_lbl_22 = dict(zip(reader_22._varlist, reader_22._lbllist))

def build_code_map(var_name, max_code=100000):
    if var_name in df_num_24.columns:
        mapping = {}
        for n, c in zip(df_num_24[var_name], df_cat_24[var_name]):
            if pd.notna(n) and pd.notna(c) and int(n) < max_code:
                mapping[int(n)] = str(c).strip()
        if mapping:
            return mapping
    if var_name in df_num_22.columns:
        lbl = var_to_lbl_22.get(var_name, '')
        if lbl and lbl in vl_22:
            return {int(k): str(v).strip() for k, v in vl_22[lbl].items() if int(k) < max_code}
    return {}

code_maps = {}
for field in active_fields:
    code_maps[field] = build_code_map(field)

del df_num_24, df_cat_24, df_num_22
gc.collect()

# Print a few examples
for f in active_fields[:5]:
    n = len(code_maps[f])
    sample = dict(list(code_maps[f].items())[:3]) if n > 0 else {}
    print(f'  {f}: {n} codes, e.g. {sample}')

Loading value labels from Stata files...
  age: 72 codes, e.g. {33: '33.0', 64: '64.0', 69: '69.0'}
  agekdbrn: 29 codes, e.g. {21: '21.0', 23: '23.0', 28: '28.0'}
  babies: 3 codes, e.g. {1: '1', 0: '0', 2: '2 or more'}
  born: 2 codes, e.g. {1: 'yes', 2: 'no'}
  childs: 9 codes, e.g. {2: '2.0', 0: '0.0', 3: '3.0'}


## 4. Load GSS data, filter D/R, create stratification bins

In [ ]:
df_gss = pd.read_csv(GSS_DATA, low_memory=False)
print(f'Loaded: {df_gss.shape[0]} respondents, {df_gss.shape[1]} columns')
print(f'Years: {sorted(df_gss["year"].unique())}')

# Filter to Democrats and Republicans
# partyid: 0=Strong D, 1=Not strong D, 2=Ind near D,
#          3=Independent, 4=Ind near R, 5=Not strong R, 6=Strong R
df_gss['party_code'] = np.nan
df_gss.loc[df_gss['partyid'].isin([0, 1, 2]), 'party_code'] = 100  # Democrat
df_gss.loc[df_gss['partyid'].isin([4, 5, 6]), 'party_code'] = 200  # Republican
df_dr = df_gss[df_gss['party_code'].notna()].copy()
print(f'D/R respondents: {len(df_dr)} '
      f'(D={int((df_dr["party_code"]==100).sum())}, R={int((df_dr["party_code"]==200).sum())})')

# Create stratification bins for sampling
# Age: bin into decades
df_dr['age_bin'] = pd.cut(df_dr['age'], bins=[0,30,40,50,60,70,100], labels=False)

# Build a combined stratification key
# Use available columns, fill missing with 'NA' for binning
strat_cols_available = [c for c in STRATIFY_COLS if c in df_dr.columns]
print(f'Stratification columns: {strat_cols_available}')

# Create strat key (coarse bins to avoid too many unique groups)
df_dr['_strat_key'] = ''
for col in strat_cols_available:
    df_dr['_strat_key'] += df_dr[col].fillna(-1).astype(int).astype(str) + '_'

n_strat_groups = df_dr['_strat_key'].nunique()
print(f'Stratification groups: {n_strat_groups}')

## 5. Stratified sampling function

In [ ]:
def stratified_sample(df, frac, seed, min_per_group=1):
    """
    Stratified sample: within each _strat_key group, sample `frac` of rows.
    Groups with fewer than min_per_group/frac members contribute all their rows.
    """
    rng = np.random.default_rng(seed)
    sampled = []
    for key, group in df.groupby('_strat_key'):
        n = max(min_per_group, int(np.ceil(len(group) * frac)))
        n = min(n, len(group))
        idx = rng.choice(group.index, size=n, replace=False)
        sampled.append(df.loc[idx])
    return pd.concat(sampled).sort_index()

# Quick test
df_sample_test = stratified_sample(df_dr, SAMPLE_FRAC, SAMPLE_SEED)
print(f'Full D/R: {len(df_dr)}, 10% stratified sample: {len(df_sample_test)}')
print(f'Sample D/R split: D={int((df_sample_test["party_code"]==100).sum())}, '
      f'R={int((df_sample_test["party_code"]==200).sum())}')
del df_sample_test

Full D/R: 6192, 10% stratified sample: 4535
Sample D/R split: D=2766, R=1769


## 6. Pre-compute demographic parts & prompt builder

In [ ]:
def precompute_demo_parts(df, code_maps, fields, labels):
    """
    Pre-compute (field_name, 'Label: value') tuples per respondent.
    Enables per-topic exclusion of overlapping survey variables.
    """
    all_parts = {}
    for idx in df.index:
        row = df.loc[idx]
        parts = []
        for field in fields:
            val = row.get(field)
            if pd.isna(val):
                continue
            code = int(val)
            label = labels.get(field, field)
            if field in code_maps and code in code_maps[field]:
                text = code_maps[field][code]
            else:
                text = str(code)
            parts.append((field, f'{label}: {text}'))
        all_parts[idx] = parts
    return all_parts

demo_parts = precompute_demo_parts(df_dr, code_maps, active_fields, all_labels)
print(f'Pre-computed parts for {len(demo_parts)} respondents')

# Show example
ex_idx = list(demo_parts.keys())[0]
print(f'\nExample (idx={ex_idx}): {len(demo_parts[ex_idx])} fields')
for field, part in demo_parts[ex_idx][:8]:
    print(f'  {part}')

Pre-computed parts for 6192 respondents

Example (idx=2): 37 fields
  Children under 6 in household: 0
  Number of children: 0.0
  Education: less than high school
  Family structure at 16: both own parents
  Health: fair
  Hours worked last week: 16.0
  Industry: furniture and home furnishings stores
  Mother's education: bachelor's


## 7. Load topics & polarization data

In [ ]:
pub_topics = pd.read_csv(PUBLIC_TOPICS)
priv_topics = pd.read_csv(PRIVATE_TOPICS)
pol_pub = pd.read_csv(PUBLIC_POL)
pol_priv = pd.read_csv(PRIVATE_POL)

# Filter polarization data
MIN_DEM, MIN_REP, MIN_TOTAL = 100, 100, 200
pol_pub = pol_pub[(pol_pub['n_dem'] >= MIN_DEM) & (pol_pub['n_rep'] >= MIN_REP) & (pol_pub['n_total'] >= MIN_TOTAL)]
pol_priv = pol_priv[(pol_priv['n_dem'] >= MIN_DEM) & (pol_priv['n_rep'] >= MIN_REP) & (pol_priv['n_total'] >= MIN_TOTAL)]

pub_valid = set(pol_pub['variable'])
priv_valid = set(pol_priv['variable'])

topic_dict_pub = {row['Variable']: row['SurveyQuestion']
                  for _, row in pub_topics.iterrows() if row['Variable'] in pub_valid}
topic_dict_priv = {row['Variable']: row['SurveyQuestion']
                   for _, row in priv_topics.iterrows() if row['Variable'] in priv_valid}

print(f'Public topics: {len(topic_dict_pub)}')
print(f'Private topics: {len(topic_dict_priv)}')
print(f'Total: {len(topic_dict_pub) + len(topic_dict_priv)}')

Public topics: 134
Private topics: 75
Total: 209


## 8. Load model

In [ ]:
model, tokenizer = load_model(MODEL_PATH, attn_implementation=ATTN_IMPL)
model_info = get_model_info(model)
print(f'Layers: {model_info["num_layers"]}, Heads: {model_info["num_heads"]}, '
      f'Head dim: {model_info["head_dim"]}')

Loading model from: /project/jevans/maxzhuyt/models/gemma-2-9b-it...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:21<00:00,  5.27s/it]

Layers: 42, Heads: 16, Head dim: 256


## 9. Core analysis functions

In [ ]:
def is_valid_response(val):
    if pd.isna(val):
        return False
    try:
        return int(float(val)) < 1000000
    except (ValueError, TypeError):
        return False


def run_topic_fast(topic_name, survey_question, df_all, demo_parts,
                   sample_frac, sample_seed, lifestyle_fields, prompt_fmt):
    """
    Run a single topic with stratified 10% sampling.
    Returns results dict or None.
    """
    t0 = time.time()

    if topic_name not in df_all.columns:
        return None

    # Filter to valid responses
    valid_mask = df_all[topic_name].apply(is_valid_response)
    df_valid = df_all[valid_mask]

    if len(df_valid) < 20:
        return None

    # Stratified sample
    df_sampled = stratified_sample(df_valid, sample_frac, sample_seed)

    n_dem = (df_sampled['party_code'] == 100).sum()
    n_rep = (df_sampled['party_code'] == 200).sum()
    if n_dem < 5 or n_rep < 5:
        return None

    # Check overlap: exclude topic from persona if it's a demographic/lifestyle field
    all_active = set(active_fields)
    exclude_field = topic_name if topic_name in all_active else None

    # Build prompts
    rng = np.random.default_rng(hash(topic_name) % (2**32))
    prompts = []
    for idx in df_sampled.index:
        if exclude_field:
            parts = [p[1] for p in demo_parts[idx] if p[0] != exclude_field]
        else:
            parts = [p[1] for p in demo_parts[idx]]
        rng.shuffle(parts)
        profile = '. '.join(parts) + '.'
        prompts.append(prompt_fmt.format(profile=profile, question=survey_question))

    labels = df_sampled['party_code'].values.astype(int)

    # Extract activations
    X_heads = extract_heads_batched(
        model, tokenizer, prompts, SYSTEM_MSG,
        batch_size=BATCH_SIZE, max_length=MAX_LENGTH
    )

    # Compute metrics
    results = {'Topic': topic_name, 'n_sampled': len(df_sampled),
               'n_dem': int(n_dem), 'n_rep': int(n_rep)}

    for centroid_method in ['mean', 'median']:
        suffix = '' if centroid_method == 'mean' else '_median'
        for n_comp in PCA_DIMS:
            grid = compute_all_head_metrics_pca(
                X_heads, labels,
                group_values=(100, 200),
                n_components=n_comp,
                centroid_method=centroid_method
            )
            results[f'Avg_Mahal_PCA{n_comp}{suffix}'] = np.mean(grid)
            results[f'Max_Mahal_PCA{n_comp}{suffix}'] = np.max(grid)

    del X_heads

    elapsed = time.time() - t0
    excl = f' [excl {exclude_field}]' if exclude_field else ''
    print(f'  {topic_name}: mahal={results[f"Avg_Mahal_PCA{PCA_DIMS[0]}"]:.3f} '
          f'(n={len(df_sampled)}, {elapsed:.1f}s){excl}', flush=True)

    return results

print('Functions defined.')

Functions defined.


: 

## 10. Run all topics (sample)

In [ ]:
all_results = []

categories = {
    'public_issues': topic_dict_pub,
    #'private_life': topic_dict_priv,
}

for cat_name, topics in categories.items():
    print(f'\n{"="*60}')
    print(f'{cat_name.upper()} ({len(topics)} topics)')
    print(f'{"="*60}')

    for i, (topic_name, survey_q) in enumerate(topics.items()):
        try:
            result = run_topic_fast(
                topic_name, survey_q, df_dr, demo_parts,
                sample_frac=0.02, sample_seed=SAMPLE_SEED,
                lifestyle_fields=lifestyle_fields, prompt_fmt=ACTIVE_PROMPT_FMT
            )
            if result is not None:
                result['category'] = cat_name
                all_results.append(result)

            if (i + 1) % 20 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        except Exception as e:
            print(f'  ERROR on {topic_name}: {e}', flush=True)
            gc.collect()
            torch.cuda.empty_cache()

df_results = pd.DataFrame(all_results)
print(f'\nTotal topics analyzed: {len(df_results)}')


PUBLIC_ISSUES (134 topics)
  > Extracting with Batch Size 90...
  abdefect: mahal=0.718 (n=2030, 259.1s)
  > Extracting with Batch Size 90...
  abhlth: mahal=0.722 (n=2049, 241.0s)
  > Extracting with Batch Size 90...
  abnomore: mahal=0.733 (n=2023, 238.1s)
  > Extracting with Batch Size 90...
  abpoor: mahal=0.771 (n=2040, 240.0s)
  > Extracting with Batch Size 90...
  abrape: mahal=0.769 (n=2038, 239.8s)
  > Extracting with Batch Size 90...
  absingle: mahal=0.747 (n=2030, 239.1s)
  > Extracting with Batch Size 90...
  abhelp1: mahal=1.038 (n=622, 73.3s)
  > Extracting with Batch Size 90...
  abhelp2: mahal=1.069 (n=621, 72.9s)
  > Extracting with Batch Size 90...
  abhelp3: mahal=1.066 (n=621, 73.0s)
  > Extracting with Batch Size 90...
  abhelp4: mahal=0.976 (n=623, 73.0s)
  > Extracting with Batch Size 90...
  advfront: mahal=0.866 (n=1444, 171.2s)
  > Extracting with Batch Size 90...
  balpos: mahal=0.898 (n=575, 68.4s)
  > Extracting with Batch Size 90...
  scientbe: mahal=0.8

## 11. Correlation with survey polarization

In [ ]:
pol_data = {
    'public_issues': pol_pub,
    #'private_life': pol_priv,
}

print('\n' + '=' * 60)
print('CORRELATION WITH SURVEY POLARIZATION')
print('=' * 60)

for cat_name in ['public_issues']:
    df_cat = df_results[df_results['category'] == cat_name]
    if len(df_cat) == 0:
        continue

    df_pol = pol_data[cat_name]
    df_merged = df_cat.merge(
        df_pol[['variable', 'polarization']].rename(
            columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
        ),
        on='Topic', how='inner'
    )

    print(f'\n[{cat_name.upper()}] (n={len(df_merged)} topics)')

    for method in ['mean', 'median']:
        suffix = '' if method == 'mean' else '_median'
        for d in PCA_DIMS:
            col = f'Avg_Mahal_PCA{d}{suffix}'
            if col in df_merged.columns:
                r_p = df_merged[col].corr(df_merged['GSS_Polarization'])
                r_s = df_merged[col].corr(df_merged['GSS_Polarization'], method='spearman')
                print(f'  Mahal PCA-{d} ({method}): r={r_p:.4f}, rho={r_s:.4f}')


CORRELATION WITH SURVEY POLARIZATION


KeyError: 'category'

## 12. Summary table for comparison across runs

In [ ]:
# Compact summary for copy-pasting across test runs
d = PCA_DIMS[0]
n_fields = len(active_fields)
lifestyle_str = 'yes' if INCLUDE_LIFESTYLE else 'no'
prompt_name = 'A' if ACTIVE_PROMPT_FMT == PROMPT_FMT_A else ('B' if ACTIVE_PROMPT_FMT == PROMPT_FMT_B else 'C')

summary_rows = []
for cat_name in ['public_issues', 'private_life']:
    df_cat = df_results[df_results['category'] == cat_name]
    if len(df_cat) == 0:
        continue
    df_pol = pol_data[cat_name]
    df_m = df_cat.merge(
        df_pol[['variable', 'polarization']].rename(
            columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
        ), on='Topic', how='inner'
    )
    for method in ['mean', 'median']:
        sfx = '' if method == 'mean' else '_median'
        col = f'Avg_Mahal_PCA{d}{sfx}'
        if col in df_m.columns:
            r_p = df_m[col].corr(df_m['GSS_Polarization'])
            r_s = df_m[col].corr(df_m['GSS_Polarization'], method='spearman')
            summary_rows.append({
                'model': MODEL_NAME, 'n_fields': n_fields,
                'lifestyle': lifestyle_str, 'prompt': prompt_name,
                'sample': SAMPLE_FRAC, 'category': cat_name,
                'centroid': method, f'r_PCA{d}': round(r_p, 4),
                f'rho_PCA{d}': round(r_s, 4),
                'n_topics': len(df_m),
            })

df_summary = pd.DataFrame(summary_rows)
print('\nSummary (copy this row to compare across configs):')
print(df_summary.to_string(index=False))

# Save
from datetime import datetime
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out = Path('../results')
out.mkdir(exist_ok=True)
summary_path = out / f'test_config_{MODEL_NAME}_{ts}.csv'
df_summary.to_csv(summary_path, index=False)
df_results.to_pickle(out / f'test_config_detail_{MODEL_NAME}_{ts}.pkl')
print(f'\nSaved: {summary_path}')


Summary (copy this row to compare across configs):
                model  n_fields lifestyle prompt  sample      category centroid  r_PCA15  rho_PCA15  n_topics
Llama-3.1-8B-Instruct        11        no      A     0.1 public_issues     mean   0.2045     0.2313       134
Llama-3.1-8B-Instruct        11        no      A     0.1 public_issues   median   0.2624     0.2904       134
Llama-3.1-8B-Instruct        11        no      A     0.1  private_life     mean   0.4461     0.4425        75
Llama-3.1-8B-Instruct        11        no      A     0.1  private_life   median   0.4569     0.4496        75

Saved: ../results/test_config_Llama-3.1-8B-Instruct_20260206_171739.csv
